In [ ]:
# ============================================================================
# CELL 1: Configuration, Dependency Versions, and Domain Pack Loader
# ============================================================================
import os, sys, json, time, math, re, hashlib, threading
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple, Set, Iterator
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import torch

# Set UTF-8 encoding & suppress verbose logs
if hasattr(sys.stdout, 'reconfigure'):
    try: sys.stdout.reconfigure(encoding='utf-8')
    except Exception: pass
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

@dataclass
class SearchConfig:
    """Global configuration settings for the generic search engine."""
    embedding_model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    reranker_model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    redis_enabled: bool = True
    redis_host: str = '127.0.0.1'
    redis_port: int = 6379
    redis_db: int = 0
    candidate_pool_size: int = 30
    rerank_top_k: int = 20
    final_top_k: int = 10
    bm25_weight: float = 0.35
    dense_weight: float = 0.65
    rrf_k: int = 60
    hard_price_filter: bool = True
    hard_audience_filter: bool = True
    hard_stock_filter: bool = False
    min_cosine_threshold: float = 0.20
    allow_relaxed_fallback: bool = True

class DomainPack:
    """Encapsulates company-specific taxonomy, lexicon, and business configuration."""
    def __init__(self, data: Dict[str, Any]):
        self.data = data
        self.domain_id = data.get('domain_id', 'default_domain')
        self.name = data.get('name', 'Default Domain')
        self.version = data.get('version', '1.0.0')
        self.schema_map = data.get('schema_map', {})
        self.taxonomy_map = data.get('taxonomy_map', {})
        self.lexicon = data.get('lexicon', {})
        self.persona_config = data.get('persona_config', {})
        self.intent_config = data.get('intent_config', {})
        self.locale_config = data.get('locale_config', {})
        self.retrieval_config = data.get('retrieval_config', {})
        self.threshold_config = data.get('threshold_config', {})
        self.aliases = {k.lower(): v.lower() for k, v in self.lexicon.get('aliases', {}).items()}
        self.brand_aliases = {k.lower(): v for k, v in self.lexicon.get('brand_aliases', {}).items()}
        self.bangla_dict = self.lexicon.get('bangla_dictionary', {})
        b_digits = self.locale_config.get('bangla_digits', {})
        self.digit_trans_table = str.maketrans(b_digits)

    @classmethod
    def from_file(cls, filepath: str) -> 'DomainPack':
        with open(filepath, 'r', encoding='utf-8') as f:
            return cls(json.load(f))

    def convert_bangla_digits(self, text: str) -> str:
        return text.translate(self.digit_trans_table)

    def resolve_alias(self, token: str) -> str:
        t_low = token.lower()
        if t_low in self.aliases: return self.aliases[t_low]
        if token in self.bangla_dict: return self.bangla_dict[token]
        return token

    def resolve_brand(self, text: str) -> Optional[str]:
        t_low = text.lower()
        for variant, canonical in self.brand_aliases.items():
            if re.search(r'\b' + re.escape(variant) + r'\b', t_low):
                return canonical
        return None

# Self-test Cell 1
cfg = SearchConfig()
print(f'[Cell 1 OK] SearchConfig initialized. Device: {cfg.device} | PyTorch: {torch.__version__}')


In [ ]:
# ============================================================================
# CELL 2: Product Schema, Normalization, Data Profiling, and Validation
# ============================================================================
@dataclass
class CanonicalProduct:
    """Standard canonical 20-field product schema with provenance tracking."""
    product_id: str
    title: str
    price: float
    sku: str = ''
    description: str = ''
    category_path: str = ''
    category_leaf: str = 'general'
    brand: str = ''
    color: str = ''
    material: str = ''
    gender: str = 'unisex'
    audience: str = 'adult'
    sizes: str = ''
    currency: str = 'BDT'
    stock_qty: int = 100
    rating: float = 4.0
    review_count: int = 0
    attributes: Dict[str, Any] = field(default_factory=dict)
    tags: List[str] = field(default_factory=list)
    is_active: bool = True
    updated_at: str = ''
    provenance: Dict[str, str] = field(default_factory=dict)
    size_set: Set[str] = field(default_factory=set)
    search_text: str = ''

    def __post_init__(self):
        if isinstance(self.sizes, str) and self.sizes.strip():
            self.size_set = {s.strip().upper() for s in self.sizes.split(',') if s.strip()}
        elif isinstance(self.sizes, (list, set)):
            self.size_set = {str(s).strip().upper() for s in self.sizes if str(s).strip()}
            self.sizes = ','.join(sorted(self.size_set))
        else:
            self.size_set = set(); self.sizes = ''
        g = str(self.gender).lower().strip()
        if g in ('men', 'man', 'male', 'boys', 'boy'): self.gender = 'men'
        elif g in ('women', 'woman', 'female', 'girls', 'girl'): self.gender = 'women'
        elif g in ('kids', 'child', 'children', 'baby', 'toddler'): self.gender = 'kids'
        else: self.gender = 'unisex'
        tags_str = ' '.join(self.tags) if isinstance(self.tags, list) else str(self.tags)
        attrs_str = ' '.join(f'{k} {v}' for k, v in self.attributes.items()) if isinstance(self.attributes, dict) else ''
        self.search_text = (
            f'{self.title}. Category: {self.category_leaf} ({self.category_path}). '
            f'Brand: {self.brand}. Color: {self.color}. Material: {self.material}. '
            f'Audience: {self.audience} {self.gender}. Description: {self.description}. '
            f'{tags_str} {attrs_str}'
        ).strip()

class SchemaProfiler:
    """Profiles raw catalog dataframes, reporting nulls, ID uniqueness, and price outliers."""
    @staticmethod
    def profile(df: pd.DataFrame) -> Dict[str, Any]:
        total = len(df)
        nulls = {c: int(df[c].isnull().sum()) for c in df.columns}
        id_cols = [c for c in df.columns if any(k in c.lower() for k in ('id', 'code', 'sku'))]
        uniqueness = {c: bool(df[c].nunique() == total) for c in id_cols}
        price_cols = [c for c in df.columns if any(k in c.lower() for k in ('price', 'cost', 'amount'))]
        price_stats = {}
        for c in price_cols:
            p = pd.to_numeric(df[c], errors='coerce').dropna()
            if len(p): price_stats[c] = {'min': float(p.min()), 'max': float(p.max()), 'median': float(p.median())}
        return {'total_rows': total, 'nulls': nulls, 'id_uniqueness': uniqueness, 'price_stats': price_stats}

class SchemaValidator:
    """Transforms raw data dicts or dataframes into canonical products with provenance."""
    def __init__(self, schema_map: Dict[str, str]):
        self.schema_map = schema_map

    def transform_row(self, row: Dict[str, Any]) -> CanonicalProduct:
        mapped, prov = {}, {}
        for c_field, s_col in self.schema_map.items():
            val = row.get(s_col)
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                mapped[c_field] = val; prov[c_field] = 'catalog'
            else:
                mapped[c_field] = None; prov[c_field] = 'missing'
        p_id = str(mapped.get('product_id') or 'AUTO_ID')
        title = str(mapped.get('title') or 'Untitled')
        try: price = float(mapped.get('price') or 0.0)
        except Exception: price = 0.0; prov['price'] = 'derived'
        return CanonicalProduct(
            product_id=p_id, title=title, price=price,
            sku=str(mapped.get('sku') or ''), description=str(mapped.get('description') or ''),
            category_path=str(mapped.get('category_path') or ''), category_leaf=str(mapped.get('category_leaf') or 'general'),
            brand=str(mapped.get('brand') or ''), color=str(mapped.get('color') or ''),
            material=str(mapped.get('material') or ''), gender=str(mapped.get('gender') or 'unisex'),
            audience=str(mapped.get('audience') or 'adult'), sizes=str(mapped.get('sizes') or ''),
            currency=str(mapped.get('currency') or 'BDT'), stock_qty=int(mapped.get('stock_qty') or 100),
            rating=float(mapped.get('rating') or 4.0), provenance=prov
        )

# Self-test Cell 2
val = SchemaValidator({'product_id': 'id', 'title': 'name', 'price': 'cost'})
sample_p = val.transform_row({'id': 'TEST01', 'name': 'Sample Shirt', 'cost': 1200})
print(f'[Cell 2 OK] Validated product: {sample_p.product_id} - {sample_p.title} (${sample_p.price}) | Provenance: {sample_p.provenance}')


In [ ]:
# ============================================================================
# CELL 3: Deterministic 500-Product SQLite Demo Catalog and First Domain Pack
# ============================================================================
from search_engine.catalog_generator import generate_ecommerce_catalog, build_and_save_demo_catalogs

# Deterministically generate and save catalogs (Seed 42)
build_and_save_demo_catalogs(data_dir='data')

# Load first Domain Pack (E-commerce retail pack)
demo_dp = DomainPack.from_file('config/domain_pack_ecommerce.json')

# Profile generated SQLite catalog
import sqlite3
conn_demo = sqlite3.connect('file:data/synthetic_catalog_1.sqlite?mode=ro', uri=True)
df_demo_raw = pd.read_sql_query('SELECT * FROM products', conn_demo)
conn_demo.close()
profile_res = SchemaProfiler.profile(df_demo_raw)
print(f'[Cell 3 OK] Demo Catalog 1 created: {profile_res["total_rows"]} products in SQLite!')
print(f'Categories: {len(df_demo_raw["category"].unique())} unique leaf categories.')
print(f'Price range: min {profile_res["price_stats"]["price"]["min"]} to max {profile_res["price_stats"]["price"]["max"]} BDT.')


In [ ]:
# ============================================================================
# CELL 4: CSV/SQL Connectors and Common Data Interface
# ============================================================================
from search_engine.connectors import BaseConnector, SQLiteConnector, CSVConnector, SQLAlchemyConnector

# Connect to SQLite demo catalog using read-only parameterized connector
sqlite_conn = SQLiteConnector('data/synthetic_catalog_1.sqlite', 'products', demo_dp.schema_map)
catalog_products = sqlite_conn.load_products()

# Connect to CSV demo catalog
csv_conn = CSVConnector('data/synthetic_catalog_1.csv', demo_dp.schema_map)
csv_products = csv_conn.load_products()

print(f'[Cell 4 OK] SQLiteConnector loaded {len(catalog_products)} canonical products.')
print(f'[Cell 4 OK] CSVConnector loaded {len(csv_products)} canonical products.')
print(f'Sample: {catalog_products[0].product_id} -> {catalog_products[0].title} ({catalog_products[0].price} {catalog_products[0].currency})')


In [ ]:
# ============================================================================
# CELL 5: Product Embeddings and BM25 Indexes
# ============================================================================
from search_engine.indexer import ProductCatalogIndexer, FastVectorizedBM25

print('Building BM25 and dense multilingual embedding index (paraphrase-multilingual-MiniLM-L12-v2)...')
indexer = ProductCatalogIndexer(
    products=catalog_products,
    embedder_model_name=cfg.embedding_model_name,
    device=cfg.device
)

# Quick self-test for BM25 and dense similarity
test_q_clean = 'black running shoe'
bm_scores = indexer.get_bm25_scores(test_q_clean)
qv = indexer.encode_query(test_q_clean)
dense_scores = indexer.get_dense_scores(qv)

print(f'[Cell 5 OK] Indexer built for {indexer.n_docs} products!')
print(f'Dense embedding matrix shape: {indexer.p_emb.shape}')
print(f'Top BM25 score for "{test_q_clean}": {np.max(bm_scores):.2f}')
print(f'Top Dense cosine similarity for "{test_q_clean}": {np.max(dense_scores):.3f}')


In [ ]:
# ============================================================================
# CELL 6: Redis Stores, Warm-up, Versioning, Invalidation, and LRU Fallback
# ============================================================================
from search_engine.cache import RedisSearchCache, ThreadSafeLRUCache

cache_sys = RedisSearchCache(
    domain_id=demo_dp.domain_id,
    domain_version=demo_dp.version,
    redis_host=cfg.redis_host,
    redis_port=cfg.redis_port,
    redis_db=cfg.redis_db,
    enabled=cfg.redis_enabled,
    maxsize=cfg.lru_cache_maxsize
)

# Verify 3-store caching with thread-safe LRU fallback
cache_sys.set_query('test query', 'cfg_hash', {'product_ids': ['P0001', 'P0002']}, ttl=60)
hit = cache_sys.get_query('test query', 'cfg_hash')

cache_sys.set_model_output('translation', 'juta', {'canonical': 'shoes'}, ttl=300)
model_hit = cache_sys.get_model_output('translation', 'juta')

print(f'[Cell 6 OK] Redis connected: {cache_sys.is_redis_connected} (LRU fallback active: True)')
print(f'Query cache hit test: {hit["product_ids"]}')
print(f'Model cache hit test: {model_hit["canonical"]}')


In [ ]:
# ============================================================================
# CELL 7: Bangla/Banglish Normalization and Translation Alternatives
# ============================================================================
from search_engine.nlp import QueryUnderstandingEngine, StructuredIntent

nlp_engine = QueryUnderstandingEngine(demo_dp)

sample_multilingual = [
    'gift for my father under 3000',
    'বাবার জন্য গিফট ৩০০০ টাকার মধ্যে',
    'বাচ্চাদের খেলনা ১৫০০ টাকার নিচে',
    'panjabi 2000 er niche',
    'online class er jonno',
    'kom dam e valo phone'
]

print('Evaluating Bangla/Banglish Query Parsing & Normalization:')
for q in sample_multilingual:
    parsed = nlp_engine.parse_query(q)
    print(f'\nQuery: "{q}"')
    print(f'  -> Lang: {parsed.language} | Max Price: {parsed.max_price} {parsed.currency}')
    print(f'  -> Recipient: {parsed.recipient} | Audience: {parsed.target_audience} | Gender: {parsed.target_gender}')
    print(f'  -> Clean Retrieval Text: "{parsed.retrieval_query}"')
    if parsed.affinities:
        print(f'  -> Inferred Affinities: {parsed.affinities[:4]}')

print('\n[Cell 7 OK] Multilingual normalizer and intent extractor verified.')


In [ ]:
# ============================================================================
# CELL 8: Integrated Intent-Aware Search with Decision Logs
# ============================================================================
from search_engine.pipeline import IntentAwareHybridSearchEngine, SearchResult

search_engine = IntentAwareHybridSearchEngine(
    config=cfg,
    domain_pack=demo_dp,
    products=catalog_products,
    warmup=True
)

# Execute primary research question test query
test_q = 'gift for my father under 3000'
search_res = search_engine.search(test_q, k=5)

print(f'=== Results for: "{test_q}" ===')
print(f'Latency: {search_res.latency_breakdown.get("total", 0)} ms | Cache Hit: {search_res.is_cache_hit}')
df_out = search_res.to_dataframe()
display(df_out[['product_id', 'title', 'category', 'price', 'relevance_score', 'bm25_score', 'dense_similarity', 'rerank_score', 'explanation']])

print('\n[Cell 8 OK] Hybrid intent-aware retrieval with decision log completed.')


In [ ]:
# ============================================================================
# CELL 9: Interactive Search Interface
# ============================================================================
def interactive_search(query_text: str, max_results: int = 5):
    """Runs interactive query and displays formatted results."""
    if not query_text.strip():
        print('Please enter a query.')
        return
    res = search_engine.search(query_text.strip(), k=max_results)
    print(f'\nSearch Query: "{query_text}"')
    print(f'Language: {res.intent.language} | Latency: {res.latency_breakdown.get("total", 0)} ms | Cache: {res.is_cache_hit}')
    print(f'Parsed Intent: Recipient={res.intent.recipient}, MaxPrice={res.intent.max_price}, Audience={res.intent.target_audience}')
    if res.relaxation_notes:
        print('Note:', '; '.join(res.relaxation_notes))
    df = res.to_dataframe()
    if not df.empty:
        display(df[['product_id', 'title', 'category', 'brand', 'price', 'relevance_score', 'explanation']])
    else:
        print('No matching products found.')

# Run example interactive search
interactive_search('বাবার জন্য গিফট ৩০০০ টাকার মধ্যে', max_results=5)
print('[Cell 9 OK] Interactive search interface ready.')


In [ ]:
# ============================================================================
# CELL 10: Labeled Test Generation, Metrics, and Ablation Evaluation
# ============================================================================
from search_engine.evaluator import generate_120_labeled_queries, BenchmarkEvaluator

labeled_queries = generate_120_labeled_queries()
print(f'Generated {len(labeled_queries)} labeled ground-truth evaluation queries across all categories!')

evaluator = BenchmarkEvaluator(search_engine, labeled_queries)
print('Running ablations across retrieval architectures...')
ablation_results_df = evaluator.run_all_ablations()

print('\n=== Retrieval Architecture Ablation Results ===')
display(ablation_results_df)

# Compare Conventional Dense Vector Search vs Full Intent-Aware Hybrid Search
dense_ndcg = ablation_results_df.loc[ablation_results_df['ablation'] == 'dense_only', 'ndcg_at_5'].values[0]
full_ndcg = ablation_results_df.loc[ablation_results_df['ablation'] == 'full_pipeline', 'ndcg_at_5'].values[0]
print(f'\nResearch Question Answer: Dense Only NDCG@5 = {dense_ndcg:.3f} vs Full Hybrid+Intent NDCG@5 = {full_ndcg:.3f}')
print(f'Relative NDCG Improvement: {((full_ndcg - dense_ndcg) / max(dense_ndcg, 1e-4)) * 100:.1f}%!')
print('[Cell 10 OK] 120-query evaluation completed.')


In [ ]:
# ============================================================================
# CELL 11: Latency Benchmark and Readiness Report
# ============================================================================
from search_engine.benchmark import benchmark_latencies

print('Measuring actual latencies across warm, cold, and cached queries...')
latency_stats = benchmark_latencies(search_engine)

print('\n=== Latency Benchmark Summary ===')
print(f'Hardware: {latency_stats["hardware"]["gpu_name"]} (Device: {latency_stats["hardware"]["device"]})')
print(f'Cold Query Latency (Mean): {latency_stats["cold_mean_ms"]} ms | p50: {latency_stats["cold_p50_ms"]} ms | p95: {latency_stats["cold_p95_ms"]} ms')
print(f'Cached Query Latency (Mean): {latency_stats["cached_mean_ms"]} ms | p50: {latency_stats["cached_p50_ms"]} ms | p95: {latency_stats["cached_p95_ms"]} ms')
print(f'Cache Hit Rate: {latency_stats["cache_hit_rate"]}% | Throughput (Sequential): {latency_stats["throughput_cold_qps"]} queries/sec')

print('\nPipeline Stage Breakdown (Cold Mean, ms):')
for stage, ms in latency_stats['stage_breakdown_cold_mean_ms'].items():
    print(f'  - {stage}: {ms} ms')

print('\n[Cell 11 OK] Latency benchmark completed.')


In [ ]:
# ============================================================================
# CELL 12: Second-Catalog Portability Test
# ============================================================================
from search_engine.benchmark import run_portability_test

# Verify domain portability by loading industrial equipment catalog with NO engine changes
portability_output = run_portability_test()

print(f'Portability Test Success: {portability_output["success"]}')
print(f'Loaded Second Catalog with {portability_output["catalog_size"]} items from {portability_output["domain_id"]}')
print('\nSample Portability Searches:')
for sample in portability_output['sample_searches']:
    print(f'  Query: "{sample["query"]}"')
    print(f'     Top Result: {sample["top_product"]} ({sample["category"]}) - {sample["price"]} | Latency: {sample["latency_ms"]} ms')

print('\n[Cell 12 OK] Zero-code-modification domain portability verified.')


# ============================================================================
# CELL 13: Architecture Diagram, Onboarding Checklist, Limitations, & Tuning
# ============================================================================

### System Architecture Overview

```text
 User Query (English / Bangla / Banglish)
       │
       ▼
 ┌─────────────────────────────────────────────────────────────┐
 │ 1. Query Normalization & Language Detection                 │
 │    - Bangla Digit Conversion (৩০০০ -> 3000)                 │
 │    - Price & Size Expression Regex Parsing                  │
 └──────────────────────────────┬──────────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 2. Query Cache Check (Redis / In-Process LRU)   ├──► [Cache Hit: < 5ms]
       └────────────────────────┬────────────────────────┘
                                │ Cache Miss
       ┌────────────────────────▼────────────────────────┐
       │ 3. Structured Intent Extraction                 │
       │    - Lexicon & Brand Normalization              │
       │    - Persona & Occasion Matching                │
       │    - Hard Constraints vs Inferred Affinities    │
       └────────────────────────┬────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 4. Parallel Multi-Channel Retrieval             │
       │    ├── Channel A: Vectorized BM25 Lexical       │
       │    ├── Channel B: Multilingual Dense Embedding  │
       │    └── Channel C: Intent-Product Compatibility  │
       └────────────────────────┬────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 5. Reciprocal Rank Fusion (RRF)                 │
       │    - Select Top-30 Balanced Candidates          │
       └────────────────────────┬────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 6. Hard Constraint Filtering & Relaxation       │
       │    - Strict Budget Ceiling ($price <= max_price)│
       │    - Demographic & Audience Boundaries          │
       └────────────────────────┬────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 7. Soft Boosts & Neural Cross-Encoder Reranking │
       │    - Brand (+14), Color (+7), Category (+10)    │
       │    - Cross-Encoder Semantic Pair Scoring        │
       └────────────────────────┬────────────────────────┘
                                │
       ┌────────────────────────▼────────────────────────┐
       │ 8. Final Ranked Results + Decision Logs         │
       │    - Update Query Cache                         │
       └─────────────────────────────────────────────────┘
```

---

### Company Server Onboarding Checklist (16 Steps)

1. **Read-Only Credentials**: Connect to company database using read-only credentials.
2. **Schema Profiling**: Run `SchemaProfiler.profile(company_df)` to inspect columns, nulls, and outliers.
3. **Proposed `schema_map`**: Review and confirm the auto-suggested column mapping.
4. **Data Normalization**: Transform products using `SchemaValidator`.
5. **Taxonomy Mapping**: Map company category paths into parent groups and leaf nodes.
6. **Build Domain Pack**: Save JSON configuration in `config/domain_pack_<company>.json`.
7. **Configure Lexicon & Aliases**: Add company brand spellings, Bengali synonyms, and product nicknames.
8. **Persona & Intent Mapping**: Map buyer personas (father, mother, student) to actual catalog categories.
9. **Vector DB Adapter**: Connect to existing company vector database (Qdrant/Milvus/pgvector) or pre-compute embeddings.
10. **Build BM25 Index**: Generate lexical index using product titles, tags, and specifications.
11. **Seed Redis Stores**: Populate Term Store (Store A) with catalog terms and aliases.
12. **Generate Company Test Queries**: Assemble labeled company-specific test queries.
13. **Benchmark Baseline vs Proposed**: Quantify relevance improvements over conventional search.
14. **Calibrate Thresholds**: Tune `min_cosine_threshold` and `rrf_k` based on validation results.
15. **Enable Shadow Search**: Run in parallel with existing search service to compare top hits without user impact.
16. **Controlled Production Rollout**: Shift live search traffic incrementally (10% -> 50% -> 100%).

---

### Technical Limitations & Tuning Guide

- **CPU vs GPU Latency**: Dense embeddings and cross-encoders on CPU take ~120-250ms per query, whereas on T4 GPU they execute in <35ms. The Redis query cache ensures repeated queries take <5ms regardless of hardware.
- **Hard Constraints vs Inferred Affinities**: Never promote an inferred category to a hard filter. When user asks for 'gift for father', keep category open while boosting masculine/father affinities.
- **Banglish Queries**: Handled via dual-strategy: Phonetic/lexicon alias normalization + Multilingual semantic embeddings. No external LLM translation is required for standard catalog queries.
